# ContractRisk M3 — evaluación FINAL sobre el gold (cuatro filas, ablaciones, juez y RAGAS)

**Sesión nueva de Colab con GPU T4** (*Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*). No requiere claves ni APIs pagas.

Qué hace, en orden: clona el repositorio en la rama indicada, instala versiones fijadas, corre las pruebas, reconstruye el índice Chroma desde cero, y ejecuta las 12 preguntas gold congeladas (`m3-qa-gold-1.0`) en:

1. línea base sin RAG (mismo generador Qwen2.5-3B, adaptación de la fila M2),
2. RAG ingenuo (denso top-3, S07),
3. RAG avanzado (híbrido BM25+denso con RRF y filtro de ID + reranking cross-encoder),
4. sistema final (avanzado + herramienta `diferencia_valores_contratos`, agente de máximo 3 pasos),
5. dos ablaciones (solo híbrido; denso + reranking).

Cada respuesta pasa por el harness `m3-harness-1.0` (exactitud, juez, dominio, latencia) y luego se calculan las cuatro métricas RAGAS (implementación del laboratorio S10, juez Qwen2.5-3B local) sobre ingenuo, avanzado y final. Duración esperada: 30–45 min. **Devuelva el ZIP de resultados y este notebook con salidas, incluso si algo falla.** No sustituya respuestas ni errores.

In [ ]:
import os, subprocess, sys, tempfile, zipfile
from pathlib import Path
# Opción A (recomendada si la rama ya está publicada): clonar el repositorio.
# Opción B: USE_ZIP = True y subir contractrisk_m3_final_codigo.zip (generado con `git archive`).
USE_ZIP = False
REPO = 'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git'
BRANCH = 'Entrega/contractrisk-m3'
ROOT = Path('/content/contractrisk_m3_final')
if ROOT.exists():
    raise RuntimeError('Use una sesión nueva: la carpeta ya existe y la evidencia anterior no se sobrescribe')
if USE_ZIP:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Suba únicamente contractrisk_m3_final_codigo.zip')
    ROOT.mkdir(parents=True)
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        for member in z.infolist():
            if not (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve()):
                raise ValueError('Ruta no permitida en ZIP')
        z.extractall(ROOT)
    COMMIT = z.comment.decode() or 'zip-sin-commit'
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(ROOT)], check=True)
    COMMIT = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
os.chdir(ROOT)
print('Código evaluado:', COMMIT)
assert Path('m3/run_final_m3.py').is_file(), 'El código no contiene la corrida final: use la rama o ZIP correctos'

In [ ]:
def run_visible(command, log_path):
    with open(log_path, 'w', encoding='utf-8') as log:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
        code = proc.wait()
    if code:
        raise RuntimeError(f'Comando falló ({code}); consulte {log_path}')

run_visible([sys.executable, '-m', 'pip', 'install', '-r', 'm3/requirements-development-v2.txt'], 'installation.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests.log')
run_visible([sys.executable, '-c', "import sys; sys.path.insert(0,'m3'); from sqlite_support import ensure_sqlite; print(ensure_sqlite()); import torch; assert torch.cuda.is_available(), 'Active GPU T4'; print(torch.cuda.get_device_name(0))"], 'environment.log')

In [ ]:
# Corrida final. La carpeta de salida debe ser nueva.
run_visible([sys.executable, '-u', 'm3/run_final_m3.py', '--output', 'final_m3_resultados'], 'execution.log')

In [ ]:
# Tablas recalculadas desde el detalle por consulta.
from IPython.display import Markdown, display
display(Markdown(Path('final_m3_resultados/tablas_m3.md').read_text(encoding='utf-8')))

In [ ]:
# Trazas de la herramienta en el sistema final: decisiones reales del agente por pregunta.
import json
for line in Path('final_m3_resultados/harness_rows.jsonl').read_text(encoding='utf-8').splitlines():
    item = json.loads(line)
    row, rec = item['row'], item['record']
    if row['variant'] != 'sistema_final':
        continue
    decisions = [(e['step'], (e['decision'] or {}).get('action'), (e['tool_result'] or {}).get('ok'),
                  (e['tool_result'] or {}).get('operation') or (e['tool_result'] or {}).get('error'))
                 for e in rec.get('tool_trace', [])]
    out = rec.get('output') or {}
    print(row['eval_id'], row['kind'], '| llamadas:', rec.get('tool_calls'), '|', rec.get('termination'), '|', decisions)
    print('   respuesta:', out.get('respuesta'), '| abstencion:', out.get('abstencion'), '| dominio:', row['domain_pass'], row['domain_reasons'])

In [ ]:
# Ejecute también tras un fallo para conservar diagnóstico.
import shutil
from google.colab import files
result_dir = ROOT / 'final_m3_resultados'
result_dir.mkdir(exist_ok=True)
(result_dir / 'commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, result_dir / log.name)
meta = result_dir / 'metadata.json'
print(meta.read_text(encoding='utf-8')[:3000] if meta.exists() else 'No se creó metadata: revisar logs.')
bundle = shutil.make_archive(str(ROOT / 'contractrisk_m3_final_resultados'), 'zip', result_dir)
files.download(bundle)

Devuelva `contractrisk_m3_final_resultados.zip` y este notebook descargado con sus salidas (*Archivo → Descargar → .ipynb*). Los resultados se versionan en `m3/results/final_m3/` y el reporte se escribe sobre ellos; no se repite la corrida para buscar mejores números.